In [8]:
import pandas as pd
from pathlib import Path

# Project paths
BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
CLEANED_DIR = BASE_DIR / "data" / "cleaned"

# Load raw datasets
cpi = pd.read_csv(RAW_DIR / "cpi.csv")
gas = pd.read_csv(RAW_DIR / "gas_prices.csv")
wages = pd.read_csv(RAW_DIR / "hourly_wages.csv")

# Preview datasets
print("CPI")
display(cpi.head())

print("Gas Prices")
display(gas.head())

print("Hourly Wages")
display(wages.head())

CPI


,observation_date,CPIAUCSL
0,1947-01-01,21.48
1,1947-02-01,21.62
2,1947-03-01,22.00
3,1947-04-01,22.00
4,1947-05-01,21.95


Gas Prices


,observation_date,GASREGW
0,1990-08-20,1.191
1,1990-08-27,1.245
2,1990-09-03,1.242
3,1990-09-10,1.252
4,1990-09-17,1.266


Hourly Wages


,observation_date,AHETPI
0,1964-01-01,2.50
1,1964-02-01,2.50
2,1964-03-01,2.51
3,1964-04-01,2.52
4,1964-05-01,2.52


In [9]:
# Standardize column names
cpi = cpi.rename(columns={
    "observation_date": "date",
    "CPIAUCSL": "cpi"
})

gas = gas.rename(columns={
    "observation_date": "date",
    "GASREGW": "gas_price"
})

wages = wages.rename(columns={
    "observation_date": "date",
    "AHETPI": "hourly_wage"
})

# Convert dates
cpi["date"] = pd.to_datetime(cpi["date"])
gas["date"] = pd.to_datetime(gas["date"])
wages["date"] = pd.to_datetime(wages["date"])

# Convert values to numeric
cpi["cpi"] = pd.to_numeric(cpi["cpi"], errors="coerce")
gas["gas_price"] = pd.to_numeric(gas["gas_price"], errors="coerce")
wages["hourly_wage"] = pd.to_numeric(wages["hourly_wage"], errors="coerce")

In [10]:
# Convert weekly gas prices to monthly averages
gas_monthly = (
    gas
    .set_index("date")
    .resample("MS")["gas_price"]
    .mean()
    .reset_index()
)

gas_monthly.head()

,date,gas_price
0,1990-08-01,1.2180
1,1990-09-01,1.2580
2,1990-10-01,1.3354
3,1990-11-01,1.3240
4,1990-12-01,1.3410


In [11]:
# Merge monthly datasets
monthly = (
    cpi
    .merge(gas_monthly, on="date", how="left")
    .merge(wages, on="date", how="left")
)

# Add date fields
monthly["year"] = monthly["date"].dt.year
monthly["month"] = monthly["date"].dt.month
monthly["month_name"] = monthly["date"].dt.month_name()

monthly.head()

,date,cpi,gas_price,hourly_wage,year,month,month_name
0,1947-01-01,21.48,NaN,NaN,1947,1,January
1,1947-02-01,21.62,NaN,NaN,1947,2,February
2,1947-03-01,22.00,NaN,NaN,1947,3,March
3,1947-04-01,22.00,NaN,NaN,1947,4,April
4,1947-05-01,21.95,NaN,NaN,1947,5,May


In [12]:
# Create analysis-ready dataset with all key metrics available
analysis_monthly = monthly.dropna(subset=["cpi", "gas_price", "hourly_wage"]).copy()

# Recalculate YoY metrics after filtering
analysis_monthly["cpi_yoy_pct"] = analysis_monthly["cpi"].pct_change(periods=12) * 100
analysis_monthly["gas_yoy_pct"] = analysis_monthly["gas_price"].pct_change(periods=12) * 100
analysis_monthly["wage_yoy_pct"] = analysis_monthly["hourly_wage"].pct_change(periods=12) * 100

# Inflation-adjusted wage measure
analysis_monthly["real_wage"] = analysis_monthly["hourly_wage"] / analysis_monthly["cpi"] * 100

# Normalize real wage to 100 at first valid point
first_valid_real_wage = analysis_monthly["real_wage"].dropna().iloc[0]
analysis_monthly["real_wage_index"] = analysis_monthly["real_wage"] / first_valid_real_wage * 100

analysis_monthly.head()

,date,cpi,gas_price,hourly_wage,year,month,month_name,cpi_yoy_pct,gas_yoy_pct,wage_yoy_pct,real_wage,real_wage_index
523,1990-08-01,131.6,1.2180,10.24,1990,8,August,NaN,NaN,NaN,7.781155,100.000000
524,1990-09-01,132.5,1.2580,10.27,1990,9,September,NaN,NaN,NaN,7.750943,99.611733
525,1990-10-01,133.4,1.3354,10.30,1990,10,October,NaN,NaN,NaN,7.721139,99.228706
526,1990-11-01,133.7,1.3240,10.32,1990,11,November,NaN,NaN,NaN,7.718773,99.198298
527,1990-12-01,134.2,1.3410,10.34,1990,12,December,NaN,NaN,NaN,7.704918,99.020236


In [13]:
# Export cleaned datasets
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

full_output_path = CLEANED_DIR / "cost_of_living_monthly_full.csv"
analysis_output_path = CLEANED_DIR / "cost_of_living_monthly_analysis.csv"

monthly.to_csv(full_output_path, index=False)
analysis_monthly.to_csv(analysis_output_path, index=False)

print(f"Full cleaned dataset exported to: {full_output_path}")
print(f"Analysis-ready dataset exported to: {analysis_output_path}")

Full cleaned dataset exported to: ..\data\cleaned\cost_of_living_monthly_full.csv
Analysis-ready dataset exported to: ..\data\cleaned\cost_of_living_monthly_analysis.csv
